In [2]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]




[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# **Part 1: Extract data to match the financial variables which is useful for calcuating the financial ratio to perform Comparable Company Analysis (CCA)**
1. Input all the financial variables extracted from the financial statements into the template excel file named 01a_3 statement model_TEMPLATE
2. Rename the excel file name to "{TICKER} 3 statement model.xlsx"
3. Input the ticker, your name, the most recent financial year and the number of years of the financial reports you have
4. Install openpyxl and pandas before processing the code
5. You may feel free the edit the TAXONOMY MAPPING REGEX DICTIONARIES section of the code below to identify the financial variables easier

*Note: If you find it difficult to have Python extract the financial variable, you may handle it manually at the template excel file named 01b_Model_Inputs_SAMPLE and start processing Part 2 of the code below.*

5. It is normal for Python to find out some of the financial variables cannot be identified. You may refer to "Audit_Working_Paper_{TICKER}.xlsx" for details and manually fix the spreadsheets at "Model_Inputs_{TICKER}.xlsx"

In [102]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill
import re
from datetime import datetime

# ==============================================================================
# CONSTANTS & CONFIGURATION
# ==============================================================================
PATH = "0052.HK 3 statement model.xlsx"
TICKER = "0052.HK"
PREPARED_BY = "Austin"
FINAL_YEAR = 2026
NUM_YEARS = 6

YEAR_COLUMNS = list(range(FINAL_YEAR - NUM_YEARS + 1, FINAL_YEAR + 1))

# ==============================================================================
# TAXONOMY MAPPING REGEX DICTIONARIES (FINANCIAL ACCOUNTING STANDARD)
# ==============================================================================

KEYWORDS_IS = {
    r"^(?!.*(?:other|cost of)).*(?:revenue|sales|turnover)": "Revenue",
    r"cost of good sold|cost of inventory sold|cost of inventories sold|cost of sales|cogs|cost of materials?": "COGS",
    r"gross profit|gross margin": "Gross Profit",
    r"depreciation|depreciation and amortisation|depreciation and amortization|depreciation & amortisation|depreciation & amortization|amortisation": "Depreciation & Amortisation",
    r"operating income|operating profit|profit from operations": "EBIT",
    r"financial income|finance income|interest revenue|financial revenue|finance revenue|interest income": "Finance Income",
    r"financial costs?|finance costs?|interest expense": "Finance Cost",
    r"income before income taxe?s?|income before taxe?s?|profit before income taxe?s?|profit before taxe?s?|ebt": "EBT",
    r"income taxe?s? expense|income taxe?s?|taxe?s?": "Income Tax",
    r"income after taxe?s? exclude? minority?|net income exclude? minority?|profit exclude? minority?|income after taxe?s? attributable to equity shareholders|income after taxe?s? attributable to equity holders|net income attributable to equity holders|profit attributable to equity holders|net income attributable to equity shareholders|profit attributable to equity shareholders": "Net Profit - before Non-controlling Interest",
    r"profit for the year|net profit|net income|income after taxe?s?": "Net Profit - after Non-controlling Interest"
}

KEYWORDS_BS = {
    r"total non-?current asset|total non current asset|total nca": "Total Non-current assets",
    r"inventor|stock": "Inventories",
    r"trade debtors?|debtors?|trade receivables?": "Trade Receivables",
    r"accounts? receivables?|debtors?|other debtors?|other receivables?": "Receivables",
    r"cash|cash and cash equivalents?": "Cash at Year End",
    r"short-?term bank deposits?|short term bank deposits?|bank deposits? \(less than 3 months?\)|bank deposits? \(< ?3 months?\)|deposits? with banks?|bank deposits? with maturity over three months?|non-pledged? time deposits \(less than 3 months?\)|non-pledged? time deposits \(< ?3 months?\)|time deposits?": "Bank Deposits (< 3 months)",
    r"total current asset|total current asset|total ca": "Total Current assets",
    r"trade creditors?|creditors?|trade payables?": "Trade Payables",
    r"accounts? payables?|creditors?|other creditors?|other payables?": "Payables",
    r"bank borrowings?|bank overdraft|borrowings?": "Bank borrowings",
    r"lease liab|lease pay": "Lease Liabilities",
    r"total current liabilit|total cl": "Total Current liabilities",
    r"total non-?current liabilit|total non current liabilit|total ncl": "Total Non-current liabilities",
    r"total equity? exclude? minority?|equity? exclude? minority?|total equity? attributable to equity holders|total equity? attributable to equity shareholders|equity? attributable to equity holders|equity? attributable to equity shareholders": "Total equity - before Non-controlling Interest",
    r"equity|total equity?": "Total equity - after Non-controlling Interest"
}

KEYWORDS_CF = {
    r"dividends? pa": "Dividends paid",
}

KEYWORDS_NOTES = {
    r"depreciation|depreciation and amortisation|depreciation and amortization|depreciation & amortisation|depreciation & amortization": "Depreciation & Amortisation",
    r"(?:other )ppe add(?:itions)|add(?:itions) for (?:other )ppe": "Capex - PPE Additions",
    r"trade debtors?|debtors?|trade receivables?": "Trade Receivables",
    r"other debtors?|other receivables?": "Other Receivables",
    r"trade creditors?|creditors?|trade payables?": "Trade Payables",
    r"other creditors?|other payables?": "Other Payables",
    r"bank deposits \(within 3 months\)|deposits? with banks?|bank deposits? with maturity over three months?|short-term bank deposits?|short term bank deposits?": "Bank Deposits (< 3 months)",
    r"cash at bank|cash on hand|cash and bank balances?": "Cash & Cash Equivalents",
}

# ==============================================================================
# EXCEL SHEET FORMATTING ENGINE
# ==============================================================================
def apply_excel_metadata_and_formatting(filename):
    """Applies corporate standard styling, metadata disclosure, and cell autofit."""
    wb = openpyxl.load_workbook(filename)
    for sheet in wb.worksheets:
        
        for row in sheet.iter_rows(min_row=1, max_row=sheet.max_row, min_col=1, max_col=sheet.max_column):
            for cell in row:
                if cell.value in ["Unclassified (Requires Manual Review)", 'No Rule Matched']:
                    cell.fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")
                if str(cell.value).endswith(( " | ⚠️ RISK EXPOSURE: Duplicate Taxonomy Entry Detected!", "(DUPLICATE EXCEPTION)")):
                    cell.fill = PatternFill(start_color="FFC000", end_color="FFC000", fill_type="solid")
        
        # Define Audit Trail Metadata Rows
        idx_source = sheet.max_row + 1
        idx_timestamp = sheet.max_row + 2

        # Styles Definition
        meta_font = Font(name="Calibri", size=11, italic=True, color="595959")
        meta_fill = PatternFill(start_color="00F2F2F2", end_color="00F2F2F2", fill_type="solid")

        # Write Disclosure
        sheet.cell(row=idx_source, column=1, value="Source: Bloomberg Terminal").font = meta_font
        sheet.cell(row=idx_source, column=1).fill = meta_fill

        timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        sheet.cell(row=idx_timestamp, column=1, value=f"Processed by: {PREPARED_BY} | Timestamp: {timestamp_str}").font = meta_font
        sheet.cell(row=idx_timestamp, column=1).fill = meta_fill

        # Dynamic Column Width Optimization
        for col in sheet.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            sheet.column_dimensions[col[0].column_letter].width = max(max_len + 3, 12)
        
    wb.save(filename)
    wb.close()

# ==============================================================================
# CORE PROCESSING ENGINE
# ==============================================================================
def process_financial_statement(sheet_name, mapping_dict, audit_writer, model_writer):
    """Executes financial data extraction, taxonomy mapping, and variance filtering."""
    print(f"[LOG] Initiating ETL Pipeline for Sheet: {sheet_name}")
    
    # Data Ingestion & Row Header Realignment
    df = pd.read_excel(PATH, sheet_name=sheet_name)
    df = df.drop([0], axis=0)
    
    line_item_header = sheet_name
    df.columns = [line_item_header] + YEAR_COLUMNS
    df[YEAR_COLUMNS] = df[YEAR_COLUMNS].apply(pd.to_numeric, errors='coerce').fillna(0)
    
    # Dynamic Variance Filtering (Eliminate Rows with Zero Activity Across All Period)
    total_period_activity = df[YEAR_COLUMNS].abs().sum(axis=1)
    df = df[total_period_activity > 0].copy()
    
    # Taxonomy Mapping & Standardized Audit Trail Initialization
    df['DCF_Target_Item'] = 'Unclassified (Requires Manual Review)'
    df['Audit_Trail'] = 'No Rule Matched'
    
    # Vectorized string conversion for high-performance matching
    normalized_names = df[line_item_header].astype(str).str.lower().str.strip()
    
    # Standardized Keyword Regex Ingestion Loop
    for regex_pattern, standardized_item in mapping_dict.items():
        match_mask = normalized_names.str.contains(regex_pattern, regex=True, na=False)
        
        # Apply mapping to rows that haven't been mapped yet (First-match precedence)
        unmapped_mask = df['DCF_Target_Item'] == 'Unclassified (Requires Manual Review)'
        active_mask = match_mask & unmapped_mask
        
        df.loc[active_mask, 'DCF_Target_Item'] = standardized_item
        df.loc[active_mask, 'Audit_Trail'] = f"Automated Match: {standardized_item}"
                
    # Duplication & Overlap Control Risk Assessment
    is_mapped = df['DCF_Target_Item'] != 'Unclassified (Requires Manual Review)'
    is_duplicate = df.duplicated(subset=['DCF_Target_Item'], keep=False)
    
    warning_mask = is_mapped & is_duplicate
    df.loc[warning_mask, 'Audit_Trail'] += " | ⚠️ RISK EXPOSURE: Duplicate Taxonomy Entry Detected!"
    df.loc[warning_mask, 'DCF_Target_Item'] += " (DUPLICATE EXCEPTION)"
    
    # Export Comprehensive Audit Working Paper
    df.to_excel(audit_writer, sheet_name=sheet_name, index=False)
    
    # Export Aggregated Clean Inputs for Financial Valuation Model
    classified_df = df[df['DCF_Target_Item'] != 'Unclassified (Requires Manual Review)']
    output_cols = ['DCF_Target_Item', sheet_name] + YEAR_COLUMNS
    model_inputs = classified_df[output_cols]
    model_inputs.to_excel(model_writer, sheet_name=sheet_name, index=False)
    print(f"[SUCCESS] Exported standardized outputs for {sheet_name}.\n")
    
# ==============================================================================
# PIPELINE EXECUTION AUTOMATION
# ==============================================================================
if __name__ == "__main__":
    tasks = [
        {"sheet_name": "Notes", "keywords": KEYWORDS_NOTES},
        {"sheet_name": "Income Statement", "keywords": KEYWORDS_IS}, 
        {"sheet_name": "Balance Sheet", "keywords": KEYWORDS_BS},
        {"sheet_name": "Cash Flow", "keywords": KEYWORDS_CF}
    ]
    
    # Master outputs filenames
    master_audit_file = f"Audit_Working_Paper_{TICKER}.xlsx"
    master_model_file = f"Model_Inputs_{TICKER}.xlsx"
 
    # Initialize single Excel writers using context managers
    with pd.ExcelWriter(master_audit_file, engine='openpyxl') as audit_writer, pd.ExcelWriter(master_model_file, engine='openpyxl') as model_writer:
        for task in tasks:
            process_financial_statement(task["sheet_name"], task["keywords"], audit_writer, model_writer)
    
    apply_excel_metadata_and_formatting(master_audit_file)
    apply_excel_metadata_and_formatting(master_model_file)
    print("[COMPLETE] Processed master workbooks successfully consolidated into tabs!")

[LOG] Initiating ETL Pipeline for Sheet: Notes
[SUCCESS] Exported standardized outputs for Notes.

[LOG] Initiating ETL Pipeline for Sheet: Income Statement
[SUCCESS] Exported standardized outputs for Income Statement.

[LOG] Initiating ETL Pipeline for Sheet: Balance Sheet
[SUCCESS] Exported standardized outputs for Balance Sheet.

[LOG] Initiating ETL Pipeline for Sheet: Cash Flow
[SUCCESS] Exported standardized outputs for Cash Flow.

[COMPLETE] Processed master workbooks successfully consolidated into tabs!


# **Part 2: Calculate the financial ratios to perform the Comparable Company Analysis (CCA)**
1. You are required to obtain the "Model_Inputs_{TICKER}.xlsx" or input all information to the "01b_Model_Inputs_SAMPLE" file to process Part 2
2. Rename your file to "Model_Inputs_{TICKER}.xlsx"
3. Input the ticker, your name, the most recent financial year and the number of years of the financial reports you have
4. Install openpyxl and pandas before processing the code
5. The Python code will help you check whether all necessary information is presented to process the code. If some of the financial variables are missing, it will automatically set that variables into 0

**Enjoy the automation!**

In [17]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill
import re
from datetime import datetime
from openpyxl.utils import get_column_letter

# ==============================================================================
# CONSTANTS & CONFIGURATION
# ==============================================================================
TICKER = "0052.HK"
PATH = "Model_Inputs_" + TICKER + ".xlsx"
PREPARED_BY = "Austin"
FINAL_YEAR = 2026 # year for the last reporting date of the year
NUM_YEARS = 6 # please input the number of years of the financial report you owned 

# # ==============================================================================
# # EXCEL SHEET FORMATTING ENGINE
# # ==============================================================================
def apply_excel_metadata_and_formatting(TICKER):
    """Applies corporate standard styling, metadata disclosure, and cell autofit."""
    wb = openpyxl.load_workbook(f"Financial_Ratio_{TICKER}.xlsx")
    ws = wb.active
    for sheet in wb.worksheets:
          
        # Define Audit Trail Metadata Rows
        idx_timestamp = sheet.max_row + 1

        # Styles Definition
        meta_font = Font(name="Calibri", size=11, italic=True, color="595959")
        meta_fill = PatternFill(start_color="00F2F2F2", end_color="00F2F2F2", fill_type="solid")

        timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        sheet.cell(row=idx_timestamp, column=1, value=f"Processed by: {PREPARED_BY} | Timestamp: {timestamp_str}").font = meta_font
        sheet.cell(row=idx_timestamp, column=1).fill = meta_fill

        for row in ws.iter_rows(min_row=3, max_row=ws.max_row, min_col=1, max_col=NUM_YEARS):
            metric_name = row[0].value
            metric_name_in_percent = ["Net Income Margin","Return on Equity","Return on Assets","EBITDA Margin"]
            if metric_name in metric_name_in_percent:
                for cell in row[1:NUM_YEARS]:
                    if cell.value and str(cell.value).startswith('='):
                        cell.number_format = '0.00%'
            else:
                for cell in row[1:NUM_YEARS]:
                    if cell.value and str(cell.value).startswith('='):
                        cell.number_format = '0.00'

        # Dynamic Column Width Optimization
        for col in sheet.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            sheet.column_dimensions[col[0].column_letter].width = max(max_len + 1, 12)

    wb.active.insert_rows(idx=1, amount=1)
    sheet.cell(row=1, column=1, value = f"{TICKER} Financial Ratio Analysis").font=Font(name="Calibri", size=14, bold=True)
        
    wb.save(f"Financial_Ratio_{TICKER}.xlsx")
    wb.close()


def generate_pandas_ratios(TICKER):
    # Structure the raw data into a clean dictionary
    df_is = pd.read_excel(PATH, sheet_name="Income Statement").drop("Income Statement", axis=1).head(-2)
    df_bs = pd.read_excel(PATH, sheet_name="Balance Sheet").drop("Balance Sheet", axis=1).head(-2)
    df_cf = pd.read_excel(PATH, sheet_name="Cash Flow").drop("Cash Flow", axis=1).head(-2)
    df_notes = pd.read_excel(PATH, sheet_name="Notes").drop("Notes", axis=1).head(-2)
    df = pd.concat([df_is, df_bs, df_cf, df_notes], ignore_index=True,sort=False)
    df = df.drop_duplicates().groupby("DCF_Target_Item",sort=False).sum()
    df = df.abs()
    
    # Build the primary DataFrame and index it by Metric Name
    df_check = ["Revenue","COGS","Depreciation & Amortisation","EBIT","Finance Cost","EBT","Income Tax","Net Profit - after Non-controlling Interest",
                "Total Non-current assets","Inventories", "Trade Receivables","Other Receivables","Bank Deposits (< 3 months)","Cash & Cash Equivalents","Total Current assets",
                "Bank borrowings","Trade Payables","Other Payables", "Lease Liabilities","Total Current liabilities","Total Non-current liabilities","Total equity - after Non-controlling Interest"]
    for line_item in df_check:
        try:
            df.loc[line_item]
        except KeyError:
            if line_item == "EBIT":
                df.loc[line_item] = df.loc["EBT"] + df.loc["Finance Cost"]
                print(line_item + " not found. Set it to EBT + Finance Cost. Please manually check the line item if needed.")
            else:
                print(line_item + " not found. Set it to 0.")
                df.loc[line_item] = 0
    
    df_base = pd.DataFrame(df)
    df_base.loc["Income Tax"] = df_base.loc["EBT"] - df_base.loc["Net Profit - after Non-controlling Interest"]
    
    # Create an empty DataFrame to hold calculated ratios
    ratio_metrics = ["Dupont Analysis",
                     "Net Income Margin","Asset Turnover","Equity Multiplier",
                     "Profitability Analysis",
                     "Return on Equity","Return on Assets","EBITDA Margin",
                     "Short term Analysis",
                     "AR Turnover","Days' AR","Inventory Turnover","Days' Inventory","AP Turnover","Days' AP","Cash Conversion Cycle",
                     "Liquidity Analysis",
                     "Current ratio","Quick ratio","Cash ratio",
                     "Solvency",
                     "Total Debt", "Debt-to-equity Ratio","Interest Coverage Ratio"]
    
    ratio_metric = ["Net Income Margin","Asset Turnover","Equity Multiplier",
                    "Return on Equity","Return on Assets","EBITDA Margin",
                    "AR Turnover","Days' AR","Inventory Turnover","Days' Inventory","AP Turnover","Days' AP","Cash Conversion Cycle",
                    "Current ratio","Quick ratio","Cash ratio",
                    "Total Debt", "Debt-to-equity Ratio","Interest Coverage Ratio"]
    
    df_ratios = pd.DataFrame(index=ratio_metrics, columns=df_base.columns)

    # Perform vector calculations across all fiscal years simultaneously
    ratio_start_col = 1
    raw_start_col = ratio_start_col + 1 + NUM_YEARS 

    def get_excel_row(item_name):
        if item_name in df_base.index:
            idx = df_base.index.get_loc(item_name)
            return 2 + idx 
        return None

    # Formulas
    for i, year in enumerate(df_base.columns):       
        # Applicable when the data obtained from raw data are used
        raw_col_letter = get_column_letter(raw_start_col + 1 + i)
        prev_raw_col_letter = get_column_letter(raw_start_col + i) if i > 0 else raw_col_letter

        # Applicable when the data obtained from financial ratios are used
        left_col_letter = get_column_letter(ratio_start_col + i)
        prev_left_col_letter = get_column_letter(ratio_start_col -1 + i) if i > 1 else left_col_letter
        

        r_revenue = 1 + get_excel_row("Revenue")
        r_cogs = 1 + get_excel_row("COGS")
        r_depre_and_amort = 1 + get_excel_row("Depreciation & Amortisation")
        r_ebit = 1 + get_excel_row("EBIT")
        r_finance_cost = 1 + get_excel_row("Finance Cost")
        r_ebt = 1 + get_excel_row("EBT")
        r_income_tax = 1 + get_excel_row("Income Tax")
        r_net_profit = 1 + get_excel_row("Net Profit - after Non-controlling Interest")
        r_noncurrent_asset = 1 + get_excel_row("Total Non-current assets")
        r_inventory = 1 + get_excel_row("Inventories")
        r_trade_rec = 1 + get_excel_row("Trade Receivables")
        r_other_rec = 1 + get_excel_row("Other Receivables")
        r_bank_deposits = 1 + get_excel_row("Bank Deposits (< 3 months)")
        r_cash = 1 + get_excel_row("Cash & Cash Equivalents")
        r_current_asset = 1 + get_excel_row("Total Current assets")
        r_bank_borrow = 1 + get_excel_row("Bank borrowings")
        r_trade_pay = 1 + get_excel_row("Trade Payables")
        r_other_pay = 1 + get_excel_row("Other Payables")
        r_lease_liab = 1 + get_excel_row("Lease Liabilities")
        r_current_liab = 1 + get_excel_row("Total Current liabilities")
        r_noncurrent_liab = 1 + get_excel_row("Total Non-current liabilities")
        r_equity = 1 + get_excel_row("Total equity - after Non-controlling Interest")


        df_ratios.at["Net Income Margin", year] = f"={raw_col_letter}{r_net_profit}/{raw_col_letter}{r_revenue}"
        df_ratios.at["Asset Turnover", year] = f"={raw_col_letter}{r_revenue}/(({raw_col_letter}{r_noncurrent_asset}+{raw_col_letter}{r_current_asset}+{prev_raw_col_letter}{r_noncurrent_asset}+{prev_raw_col_letter}{r_current_asset})/2)"
        df_ratios.at["Equity Multiplier", year] = f"=((({raw_col_letter}{r_noncurrent_asset}+{raw_col_letter}{r_current_asset}+{prev_raw_col_letter}{r_noncurrent_asset}+{prev_raw_col_letter}{r_current_asset})/2))/(({raw_col_letter}{r_equity}+{prev_raw_col_letter}{r_equity})/2)"
        df_ratios.at["Return on Equity", year] = f"={raw_col_letter}{r_net_profit}/(({raw_col_letter}{r_equity}+{prev_raw_col_letter}{r_equity})/2)"
        df_ratios.at["Return on Assets", year] = f"={raw_col_letter}{r_net_profit}/(({raw_col_letter}{r_noncurrent_asset}+{raw_col_letter}{r_current_asset}+{prev_raw_col_letter}{r_noncurrent_asset}+{prev_raw_col_letter}{r_current_asset})/2)"
        df_ratios.at["EBITDA Margin", year] = f"=({raw_col_letter}{r_ebit}+{raw_col_letter}{r_depre_and_amort})/{raw_col_letter}{r_revenue}"

        df_ratios.at["AR Turnover", year] = f"={raw_col_letter}{r_revenue}/((({raw_col_letter}{r_trade_rec}+{raw_col_letter}{r_other_rec})+({prev_raw_col_letter}{r_trade_rec}+{prev_raw_col_letter}{r_other_rec}))/2)"
        r_art_self = 1 + ratio_metrics.index("AR Turnover") + 2
        df_ratios.at["Days' AR", year] = f"=360/{left_col_letter}{r_art_self}"

        df_base.loc["Inventory Turnover"] = df_base.loc["COGS"] / ((df_base.loc["Inventories"]+df_base.loc["Inventories"].shift(1))/2)
        
        df_ratios.at["Inventory Turnover", year] = f"={raw_col_letter}{r_cogs}/(({raw_col_letter}{r_inventory}+{prev_raw_col_letter}{r_inventory})/2)"
        if (df_base.loc["Inventory Turnover"].any() == 0):
            df_ratios.at["Days' Inventory",year] = 0
        else:
            r_inv_self = 1 + ratio_metrics.index("Inventory Turnover") + 2
            df_ratios.at["Days' Inventory",year] = f"=360/{left_col_letter}{r_inv_self}"

        df_ratios.at["AP Turnover", year] = f"={raw_col_letter}{r_cogs}/((({raw_col_letter}{r_trade_pay}+{raw_col_letter}{r_other_pay})+({prev_raw_col_letter}{r_trade_pay}+{prev_raw_col_letter}{r_other_pay}))/2)"

        df_base.loc["AP Turnover"] = df_base.loc["COGS"] / (( df_base.loc["Trade Payables"]+df_base.loc["Other Payables"]+df_base.loc["Trade Payables"].shift(1)+df_base.loc["Other Payables"].shift(1))/2)
        
        if (df_base.loc["AP Turnover"].any() == 0) and (df_base.at["COGS",year] == 0):
            df_ratios.at["Days' AP",year] = f"=((({raw_col_letter}{r_trade_pay}+{raw_col_letter}{r_other_pay})+({prev_raw_col_letter}{r_trade_pay}+{prev_raw_col_letter}{r_other_pay}))/2)/({raw_col_letter}{r_revenue}-{raw_col_letter}{r_ebit})*360"
        else:
            r_apt_self = 1 + ratio_metrics.index("AP Turnover") + 2
            df_ratios.at["Days' AP",year] = f"=360/{left_col_letter}{r_apt_self}"

        r_dayar_self = 1 + ratio_metrics.index("Days' AR") + 2
        r_dayinv_self = 1 + ratio_metrics.index("Days' Inventory") + 2
        r_dayap_self = 1 + ratio_metrics.index("Days' AP") + 2
        df_ratios.at["Cash Conversion Cycle",year] = f"={left_col_letter}{r_dayar_self}+{left_col_letter}{r_dayinv_self}-{left_col_letter}{r_dayap_self}"

        df_ratios.at["Current ratio",year] = f"={raw_col_letter}{r_current_asset}/{raw_col_letter}{r_current_liab}"
        df_ratios.at["Quick ratio",year] = f"=({raw_col_letter}{r_cash}+{raw_col_letter}{r_bank_deposits}+{raw_col_letter}{r_trade_rec})/{raw_col_letter}{r_current_liab}"
        df_ratios.at["Cash ratio",year] = f"=({raw_col_letter}{r_cash}+{raw_col_letter}{r_bank_deposits})/{raw_col_letter}{r_current_liab}"

        df_ratios.at["Total Debt",year] = f"={raw_col_letter}{r_bank_borrow}+{raw_col_letter}{r_lease_liab}"
        r_debt_self = 1 + ratio_metrics.index("Total Debt") + 2
        df_ratios.at["Debt-to-equity Ratio",year] = f"={left_col_letter}{r_debt_self}/{raw_col_letter}{r_equity}"
        df_ratios.at["Interest Coverage Ratio",year] = f"={raw_col_letter}{r_ebit}/{raw_col_letter}{r_finance_cost}"

    df_ratios.drop([FINAL_YEAR - NUM_YEARS + 1],axis=1, inplace=True)
    df_ratios = df_ratios.reset_index(names='Financial Ratios')
    df = df.reset_index(names='Financial Variables (in $ millions)')
    dfx = pd.DataFrame({"":[""]})

    df1 = pd.concat([df_ratios, dfx, df], ignore_index=False,sort=False, axis=1)
    
    with pd.ExcelWriter(f"Financial_Ratio_{TICKER}.xlsx", engine="openpyxl") as writer:
        df1.to_excel(writer, sheet_name="Financial Analysis", index=False)
        
    print(f"File created successfully: Financial_Ratio_{TICKER}.xlsx")

if __name__ == "__main__":
    generate_pandas_ratios(TICKER)
    apply_excel_metadata_and_formatting(TICKER)
print("[COMPLETE]")

File created successfully: Financial_Ratio_0052.HK.xlsx
[COMPLETE]
